# Instalação

In [ ]:
# Install required packages.
import os
import torch
os.environ['TORCH'] = torch.__version__
print(torch.__version__)

!pip install -q torch-scatter -f https://data.pyg.org/whl/torch-${TORCH}.html
!pip install -q torch-sparse -f https://data.pyg.org/whl/torch-${TORCH}.html
!pip install -q git+https://github.com/pyg-team/pytorch_geometric.git
!pip install dgl

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 280.2/280.2 KB 8.8 MB/s eta 0:00:00


In [ ]:
!cat /proc/cpuinfo
!df -h
!cat /proc/meminfo
!nvidia-smi
!nvidia-smi --query-gpu=memory.total

processor	: 0
vendor_id	: GenuineIntel
cpu family	: 6
model		: 79
model name	: Intel(R) Xeon(R) CPU @ 2.20GHz
stepping	: 0
microcode	: 0xffffffff
cpu MHz		: 2199.998
cache size	: 56320 KB
physical id	: 0
siblings	: 2
core id		: 0
cpu cores	: 1
apicid		: 0
initial apicid	: 0
fpu		: yes
fpu_exception	: yes
cpuid level	: 13
wp		: yes
flags		: fpu vme de pse tsc msr pae mce cx8 apic sep mtrr pge mca cmov pat pse36 clflush mmx fxsr sse sse2 ss ht syscall nx pdpe1gb rdtscp lm constant_tsc rep_good nopl xtopology nonstop_tsc cpuid tsc_known_freq pni pclmulqdq ssse3 fma cx16 pcid sse4_1 sse4_2 x2apic movbe popcnt aes xsave avx f16c rdrand hypervisor lahf_lm abm 3dnowprefetch invpcid_single ssbd ibrs ibpb stibp fsgsbase tsc_adjust bmi1 hle avx2 smep bmi2 erms invpcid rtm rdseed adx smap xsaveopt arat md_clear arch_capabilities
bugs		: cpu_meltdown spectre_v1 spectre_v2 spec_store_bypass l1tf mds swapgs taa mmio_stale_data retbleed
bogomips	: 4399.99
clflush size	: 64
cache_alignment	: 64
addres

In [ ]:
!pip install nvidia-ml-py3
!pip install ipython-autotime
%load_ext autotime

Looking in indexes: https://pypi.org/simple, https://us-python.pkg.dev/colab-wheels/public/simple/
  Preparing metadata (setup.py) ... done
  Created wheel for nvidia-ml-py3: filename=nvidia_ml_py3-7.352.0-py3-none-any.whl size=19190 sha256=dd4338b7cb5db44cfa44cf3315b4bc1aa1c4292713b16284c0c71760853acb79
  Stored in directory: /root/.cache/pip/wheels/b9/b1/68/cb4feab29709d4155310d29a421389665dcab9eb3b679b527b
Successfully built nvidia-ml-py3
Looking in indexes: https://pypi.org/simple, https://us-python.pkg.dev/colab-wheels/public/simple/
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 27.8 MB/s eta 0:00:00
time: 539 µs (started: 2023-02-22 05:40:33 +00:00)


# Utils

In [ ]:
!pip install dgl

Looking in indexes: https://pypi.org/simple, https://us-python.pkg.dev/colab-wheels/public/simple/
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.4/5.4 MB 53.3 MB/s eta 0:00:00
time: 5.7 s (started: 2023-02-22 05:42:09 +00:00)


In [ ]:
import torch
import dgl
import dgl.function as fn
from dgl.nn import GraphConv
from dgl.dataloading import GraphDataLoader
from dgl.data import AmazonCoBuyComputerDataset
from torch.utils.data import DataLoader

# Define model
class CGN(torch.nn.Module):
    def __init__(self, in_feats, h_feats, out_feats):
        super(CGN, self).__init__()
        self.conv1 = GraphConv(in_feats, h_feats)
        self.conv2 = GraphConv(h_feats, out_feats)

    def forward(self, graph, inputs):
        x = inputs
        x = self.conv1(graph, x).relu()
        x = torch.nn.functional.dropout(x, p=0.5, training=self.training)
        x = self.conv2(graph, x)
        return x

# Load the dataset
dataset = AmazonCoBuyComputerDataset()

g = dataset[0]
feat = g.ndata['feat']

# Define transformation
def transform(data):
    return data

# Create dataloader
loader = GraphDataLoader(
    dataset,
    batch_size=32,
    shuffle=True,
    drop_last=False,
)

# Create model and optimizer
model = CGN(767, 16, dataset.num_classes)
optimizer = torch.optim.Adam(model.parameters(), lr=0.01, weight_decay=5e-4)

# Train function
def train(model, optimizer, train_loader):
    model.train()
    optimizer.zero_grad()
    logits = model(train_loader, feat)
    loss = torch.nn.functional.cross_entropy(logits, batch.labels)
    loss.backward()
    optimizer.step()

# Train the model
torch.manual_seed(12345)
for epoch in range(1, 101):
    for step, subgraph in enumerate(loader):
        train(model, optimizer, subgraph)
    print(f'Epoch {epoch} done')

DGLError: ignored

time: 294 ms (started: 2023-02-22 05:58:03 +00:00)
